# Local Navigation

**Why use a state machine and not a potential field?**

A COMPLETER

### Test 1: Avoiding obstacles only

First, I program the Thymio to go forward at a constant speed of speed0=100. I then place an obstacle (Thymio's box) on the way, he must detect it, slow down and avoid it.
Here, I only test the sensors, the gains and its avoiding attitude, I don't need to know its position for now.

In [1]:
!pip install --upgrade tdmclient


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Connect your Thymio, open Thymio Suite and then execute:
import tdmclient.notebook
await tdmclient.notebook.start()

In [2]:
%%run_python 
#indicates that the following cell is sent to the Thymio

speed0 = 100       # nominal speed when going forward
obstThrL = 10      # low obstacle threshold to switch state 1->0
obstThrH = 20      # high obstacle threshold to switch state 0->1
obstSpeedGain = 5  # /100 (actual gain: 5/100=0.05)

state = 1          # 0=forward, 1=obstacle avoidance
obst = [0,0]       # measurements from left and right prox sensors

timer_period[0] = 10   # 10ms sampling time, frequency of 100 Hz, proximity sensors are actualized at a frequency of 10 Hz


@onevent # indicates to the Thymio that the function must be executed when a particular event happens, here its timer0, so the event is triggered every 10ms
def timer0():
    global prox_horizontal, motor_left_target, motor_right_target, state, obst, obstThrH, obstThrL, obstSpeedGain, speed0 

    # acquisition from the proximity sensors to detect obstacles
    obst = [prox_horizontal[0], prox_horizontal[4]] #Le programme prend seulement deux des capteurs horizontaux, le plus à gauche 0 et le plus à droite 4
    
    # tdmclient does not support yet multiple and/or in if statements:
    if state == 0: #Si en mode "suivre gradient"
        # switch from going forward to obst avoidance if obstacle detected
        if (obst[0] > obstThrH):
            state = 1
        elif (obst[1] > obstThrH):
            state = 1
    elif state == 1:
        if obst[0] < obstThrL:
            if obst[1] < obstThrL : 
                # switch from obst avoidance to going forward if obstacle got unseen
                state = 0
    if  state == 0 :
        # going forward
        motor_left_target = speed0
        motor_right_target = speed0
    else:
        # obstacle avoidance: accelerate wheel near obstacle
        motor_left_target = speed0 + obstSpeedGain * (obst[0] // 100) #On fait // 100 parce que la valeur brute des capteurs est très élevée (de 0 à 5000 environ)
        motor_right_target = speed0 + obstSpeedGain * (obst[1] // 100) # // est une division entière, donc on ne garde pas la partie décimale

In [3]:
stop()

The previous code is using only the two proximity sensors on the sides. So I'm changing this so that all proximity sensors are used, otherwise, obstacles in front might not get detected.

Also, the thresholds:

obstThrL = 10      # low obstacle threshold to switch state 1->0
obstThrH = 20      # high obstacle threshold to switch state 0->1

should be changed so that they correspond to the distance where we think we should stop/start avoiding the obstacle

In [4]:
%%run_python
#indicates that the following cell is sent to the Thymio

speed0 = 100       # nominal speed when going forward
obstThrL = 10      # low obstacle threshold to switch state 1->0
obstThrH = 20      # high obstacle threshold to switch state 0->1
obstSpeedGain = [6, 4, -2, -6, -8]    # /100
                  # gains used with front proximity sensors 0..4

state = 1          # 0=forward, 1=obstacle avoidance

timer_period[0] = 10   # 10ms sampling time, frequency of 100 Hz, proximity sensors are actualized at a frequency of 10 Hz


@onevent # indicates to the Thymio that the function must be executed when a particular event happens, here its timer0, so the event is triggered every 10ms
def timer0():
    global prox_horizontal, motor_left_target, motor_right_target, state, obstThrH, obstThrL, obstSpeedGain, speed0     
    
    # tdmclient does not support yet multiple and/or in if statements:
    if state == 0:
        for i in range(5):
            if prox_horizontal[i] > obstThrH: #if any of the proximity sensors detect an obstacle
                state = 1                     #we switch to obstacle avoidance mode
    elif state == 1:
        obstacle_gone = 1

        for i in range(5):
            if prox_horizontal[i] >= obstThrL: #if one of the 5 sensors still detects an obstacle
                obstacle_gone = 0              #then we stay in obstacle avoidance mode

        if obstacle_gone == 1:                 #if none of the sensors detect an obstacle
            state = 0                          #we switch to going forward mode
                
    if  state == 0 :
        # goal tracking: turn toward the goal
        spLeft = speed0
        spRight = speed0

        motor_left_target = spLeft
        motor_right_target = spRight

    else:
        # obstacle avoidance
        spLeft = speed0
        spRight = speed0

        for i in range(5):
            spLeft += prox_horizontal[i] * obstSpeedGain[i] // 100
            spRight += prox_horizontal[i] * obstSpeedGain[4 - i] // 100

        # motor control
        motor_left_target = spLeft
        motor_right_target = spRight

In [5]:
stop()

This code works great for convex obstacles but not for concave obstacles. For example, if you put an obstacle with a V shape in front of the robot, its left sensors will tell the robot to turn right, but the right sensors will also detect an obstacle so they will tell the robot to turn left hence the robot will be blocked.

To solve this problem, maybe we could implement a third state where when the robot detects that it is blocked, it makes an avoiding maneuver. Since the professor told us that we can put whatever shape of obstacle and wherever we want on the map, for now I leave this problem on the side and I will maybe solve it later.

### Test 2: Visual Localization and Waypoint Navigation

**Step 1: Computer Vision Integration**


In [6]:
import computer_vision as vision

# Open webcam and capture one image
with vision.Camera(index=0) as camera:
    image = camera.read()

# Analyse the image
frame = vision.analyse_frame(image)

# Display detected information
print("Thymio position:", frame.thymio)
print("Goal position:", frame.goal)
print("Detected ArUco markers:", sorted(frame.markers.keys()))

# Display annotated map
vision.show(vision.draw_frame(frame))

ModuleNotFoundError: No module named 'computer_vision'